In [2]:
import chromadb
from sentence_transformers import SentenceTransformer

/Users/ajinkya/Desktop/Projs/Construction-risk/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import json


In [5]:

from tqdm import tqdm
import openai

In [7]:
import pandas as pd

In [8]:
import numpy as np

In [9]:
df = pd.read_csv("DOB_NOW_cleaned.csv")

/var/folders/2t/khs2x02565z4jcwmj2770ym80000gn/T/ipykernel_2328/1323950220.py:1: DtypeWarning: Columns (0: applicant_license_) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("DOB_NOW_cleaned.csv")


In [11]:
with open(
    "embedding_documents.json",
    "r"
) as f:

    documents = json.load(f)

In [12]:
model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20031.22it/s]


In [13]:
client = chromadb.PersistentClient(
    path="./chroma_db"
)

In [14]:
collection = client.get_or_create_collection(
    name="construction_risk_rag"
)

In [15]:
batch_size = 128

for i in tqdm(
    range(0, len(documents), batch_size)
):
    batch_docs = documents[i:i+batch_size]
    embeddings = model.encode(
        batch_docs
    ).tolist()

    ids = [

        f"project_{i+j}"

        for j in range(len(batch_docs))
    ]

    metadatas = []

    for j in range(len(batch_docs)):

        row = df.iloc[i + j]

        metadata = {

            "project_id": f"project_{i+j}",

            "borough": str(
                row.get("borough", "UNKNOWN")
            ),

            "job_type": str(
                row.get("job_type", "UNKNOWN")
            ),

            "bin": str(
                row.get("bin", "UNKNOWN")
            )
        }

        metadatas.append(metadata)

    collection.add(

        ids=ids,

        documents=batch_docs,

        embeddings=embeddings,

        metadatas=metadatas
    )
        
print("\nALL PROJECTS STORED IN CHROMADB!")
    


100%|██████████| 441/441 [05:12<00:00,  1.41it/s]


ALL PROJECTS STORED IN CHROMADB!


In [16]:
query = """

Foundation excavation project
with structural work in Manhattan

"""

query_embedding = model.encode(
    [query]
).tolist()[0]

results = collection.query(

    query_embeddings=[query_embedding],

    n_results=5
)


for i in range(len(results["documents"][0])):

    print(f"\n================ RESULT {i+1} ================\n")

    print("DOCUMENT:\n")

    print(results["documents"][0][i])

    print("\nMETADATA:\n")

    print(results["metadatas"][0][i])

    print("\n================================================")


================ RESULT 1 ================

DOCUMENT:



    PROJECT DETAILS:

    Job Type: New Building
    Borough: MANHATTAN
    BIN: 1091736
    Block: 707
    Lot: 20

    Building Type: Other

    Initial Cost: 1000000.0

    Floor Area:
    1358960.0

    Existing Stories:
    UNKNOWN

    Proposed Stories:
    50.0

    Existing Dwelling Units:
    UNKNOWN

    Proposed Dwelling Units:
    0.0

    Filing Date:
    2026-01-26

    CONTRACTOR & SCOPE:

    Applicant Professional Title:
    PE

    Owner Business:
    517 WEST 35TH LLC

    Filing Representative:
    RELATED COMPANIES

    Building Code:
    2022

    Structural Work:
    0

    Foundation Work:
    0

    Mechanical Systems Work:
    0

    General Construction Work:
    0

    Support Of Excavation:
    No

    Special Inspection Requirement:
    Excavations - Sheeting, Shoring, and Bracing,Structural Steel - Welding,Subgrade Inspections,Subsurface Conditions – Fill Placement & In-Place Density,Subsurface Inv